In [0]:
# Release only the first history batch.
# Evaluation batches remain in staging.
batch_date = "2019-01-01"

staging_root = (
    "/Volumes/workspace/sentinel_sources/"
    "replay_staging/replay_v1/transactions"
)

landing_root = (
    "/Volumes/workspace/sentinel_sources/replay_landing"
)

source_path = f"{staging_root}/replay_batch={batch_date}"
destination_path = f"{landing_root}/replay_batch={batch_date}"

# Read the selected daily folder and verify it before copying.
batch = spark.read.parquet(source_path)

assert batch.count() == 2414, "Expected 2,414 transactions"
assert "is_fraud" not in batch.columns, (
    "Fraud labels must not enter landing"
)

# Stop if this date already exists in landing.
# Released files should remain unchanged for incremental ingestion.
existing_folders = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(landing_root)
}

assert f"replay_batch={batch_date}" not in existing_folders, (
    f"Batch {batch_date} already exists in landing. "
    "Do not overwrite or release it again."
)

# Copy the prepared daily folder without changing its contents.
copied = dbutils.fs.cp(
    source_path,
    destination_path,
    recurse=True
)

assert copied, "Batch copy did not complete successfully"

# Verify the files now available in landing.
released_batch = spark.read.parquet(destination_path)
released_count = released_batch.count()

assert released_count == 2414, (
    f"Expected 2,414 released transactions; found {released_count:,}"
)

print(f"Released batch: {batch_date}")
print(f"Verified landing transactions: {released_count:,}")
print("Evaluation batches remain unreleased.")